# 09 — Cohort balance check

The PCA plot of the combined features showed the two cohorts sitting in completely
separate regions: PC 1, the single strongest pattern in the data, is essentially
"OSF or parcel". That is a batch effect, and it opens a specific failure mode.

**The worry.** If OSF and the parcel cohort have different AD rates, then a model can
score well on the combined 309 subjects without looking at the brain at all. It just
detects which dataset a subject came from and guesses the local majority class. The
accuracy number looks fine; the result means nothing.

**What this notebook does.**

1. Prints the AD/control split within each cohort.
2. Computes the *cheat ceiling* - the accuracy and AUC obtainable from cohort
   membership alone, with no features. This is the floor any real result must clear.
3. Breaks the existing frozen+LR results down per cohort, which is the number that
   actually settles whether the combined result is real.
4. Confirms numerically how separable the cohorts are in feature space.

Nothing here retrains anything. It reads files notebooks 01-05 already wrote and runs
in a few seconds.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
from scipy.stats import fisher_exact
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

OSF, OUT = '../data/preprocessed', '../results_parcel'
MODEL, SUFFIX = 'base', 'cortical'

# Same load order as notebooks 04 and 05: OSF rows first, then parcel rows.
# The order matters because the saved predictions are indexed against it.
osf_y = np.load(f'{OSF}/labels.npy')
par_y = np.load(f'{OUT}/parcel_labels_{SUFFIX}.npy')
labels = np.concatenate([osf_y, par_y])
source = np.array(['osf'] * len(osf_y) + ['parcel'] * len(par_y))
N = len(labels)

print(f'{N} subjects (OSF {len(osf_y)} + parcel {len(par_y)})\n')
print(f'{"cohort":10s} {"n":>5s} {"AD":>5s} {"Control":>8s} {"AD rate":>9s}')
for c in ['osf', 'parcel']:
    y = labels[source == c]
    print(f'{c:10s} {len(y):5d} {int((y==1).sum()):5d} {int((y==0).sum()):8d} {y.mean():9.3f}')
y = labels
print(f'{"combined":10s} {N:5d} {int((y==1).sum()):5d} {int((y==0).sum()):8d} {y.mean():9.3f}')

309 subjects (OSF 148 + parcel 161)

cohort         n    AD  Control   AD rate
osf          148    78       70     0.527
parcel       161    77       84     0.478
combined     309   155      154     0.502


## The cheat ceiling

Below is the best a model could do knowing **only** which dataset each subject came
from. Two versions:

- **accuracy** - predict the majority class within each cohort.
- **AUC** - score every subject by their own cohort's AD rate, so all of one cohort
  outranks all of the other.

Read it like this. If the cheat ceiling is near 0.5, the cohorts are balanced and the
batch effect cannot inflate your results - the combined numbers are safe. If it is
high, then that much of your reported accuracy could come from dataset identity alone,
and the combined result needs to be reported per cohort instead.

In [2]:
rates = {c: labels[source == c].mean() for c in ['osf', 'parcel']}

# Accuracy from cohort alone: guess each cohort's majority class.
cheat_pred = np.array([int(rates[c] >= 0.5) for c in source])
cheat_acc  = accuracy_score(labels, cheat_pred)

# AUC from cohort alone: rank by the cohort's AD rate.
cheat_prob = np.array([rates[c] for c in source])
cheat_auc  = roc_auc_score(labels, cheat_prob)

# Is the AD/control split actually different between cohorts, or just noise?
table = np.array([[int(((source == c) & (labels == k)).sum()) for k in (1, 0)]
                  for c in ('osf', 'parcel')])
odds, p = fisher_exact(table)

print(f'AD rate   OSF {rates["osf"]:.3f}   parcel {rates["parcel"]:.3f}   '
      f'difference {abs(rates["osf"] - rates["parcel"]):.3f}')
print(f'Fisher exact test: p = {p:.4g}   (odds ratio {odds:.2f})\n')
print(f'CHEAT CEILING - using cohort membership only, no features:')
print(f'  accuracy {cheat_acc:.3f}')
print(f'  AUC      {cheat_auc:.3f}\n')

if p > 0.05:
    print('Cohorts are balanced on AD rate. The batch effect cannot inflate the')
    print('combined result through label imbalance. Combined numbers are trustworthy')
    print('on this axis.')
else:
    print('Cohorts differ significantly on AD rate. Any combined-cohort number must be')
    print('compared against the cheat ceiling above, not against 0.5, and is best')
    print('reported per cohort as well.')

AD rate   OSF 0.527   parcel 0.478   difference 0.049
Fisher exact test: p = 0.4261   (odds ratio 1.22)

CHEAT CEILING - using cohort membership only, no features:
  accuracy 0.524
  AUC      0.524

Cohorts are balanced on AD rate. The batch effect cannot inflate the
combined result through label imbalance. Combined numbers are trustworthy
on this axis.


## Per-cohort breakdown of the real results

The decisive check. The combined frozen+LR result is re-computed exactly as notebook 05
does it, then scored **separately within each cohort**.

A genuine result works inside each cohort on its own, where dataset identity is
constant and therefore useless as a shortcut. A result that only exists in the pooled
number, and collapses toward chance within cohorts, was riding the batch effect.

In [3]:
X = np.load(f'{OUT}/combined_frozen_{MODEL}_{SUFFIX}.npy')
assert len(X) == N, f'feature/label mismatch: {len(X)} vs {N}'

cv = StratifiedKFold(5, shuffle=True, random_state=123)
lr = LogisticRegression(max_iter=2000)
pred = cross_val_predict(lr, X, labels, cv=cv)
prob = cross_val_predict(lr, X, labels, cv=cv, method='predict_proba')[:, 1]

print(f'Frozen BrainOmni-{MODEL} + LR, 5-fold CV\n')
print(f'{"scored on":18s} {"n":>5s} {"acc":>7s} {"AUC":>7s}')
print(f'{"combined":18s} {N:5d} {accuracy_score(labels, pred):7.3f} '
      f'{roc_auc_score(labels, prob):7.3f}')
for c in ['osf', 'parcel']:
    m = source == c
    print(f'{c:18s} {m.sum():5d} {accuracy_score(labels[m], pred[m]):7.3f} '
          f'{roc_auc_score(labels[m], prob[m]):7.3f}')
print(f'{"cheat ceiling":18s} {N:5d} {cheat_acc:7.3f} {cheat_auc:7.3f}   <- beat this')

Frozen BrainOmni-base + LR, 5-fold CV

scored on              n     acc     AUC
combined             309   0.825   0.903
osf                  148   0.791   0.876
parcel               161   0.857   0.925
cheat ceiling        309   0.524   0.524   <- beat this


### Figure: per-cohort scores against the cheat ceiling

One panel per metric - never two y-scales on one axis. The dashed line is the cheat
ceiling from above: bars comfortably clear of it are real signal, bars sitting on it
are dataset identity.

In [ ]:
import matplotlib.pyplot as plt

groups = ['Combined', 'OSF\n(sensor)', 'Parcel\n(source)']
masks  = [np.ones(N, bool), source == 'osf', source == 'parcel']
accs   = [accuracy_score(labels[m], pred[m]) for m in masks]
aucs   = [roc_auc_score(labels[m], prob[m]) for m in masks]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, vals, ceil, lab in ((axes[0], accs, cheat_acc, 'Accuracy'),
                            (axes[1], aucs, cheat_auc, 'AUC')):
    x = np.arange(len(groups))
    ax.bar(x, vals, width=0.6, color='#4E79A7', zorder=3)
    ax.axhline(ceil, ls='--', lw=1.5, color='#555', zorder=4,
               label=f'cheat ceiling ({ceil:.3f})')
    ax.axhline(0.5, ls=':', lw=1, color='#999', zorder=2, label='chance (0.500)')
    for xi, v in zip(x, vals):                      # few bars, so label them all
        ax.text(xi, v + 0.012, f'{v:.3f}', ha='center', fontsize=9, color='#333')
    ax.set_xticks(x); ax.set_xticklabels(groups, fontsize=9)
    ax.set_ylim(0.4, 1.0); ax.set_ylabel(lab); ax.set_title(lab)
    ax.grid(axis='y', alpha=0.3); ax.set_axisbelow(True)
    ax.legend(fontsize=8, loc='upper right')
    for s in ('top', 'right'): ax.spines[s].set_visible(False)

fig.suptitle(f'Frozen BrainOmni-{MODEL} + LR, scored within each cohort', fontweight='bold')
plt.tight_layout()
fname = f'cohort_balance_{MODEL}.png'
plt.savefig(f'{OUT}/{fname}', dpi=150, bbox_inches='tight'); plt.show()
print('saved', fname)

## How separable are the cohorts, really?

The PCA plot showed the split visually. This puts a number on it: train a classifier to
predict *cohort* (not disease) from the same features. An AUC near 1.0 confirms the
batch effect is as strong as the picture suggested.

In [4]:
is_parcel = (source == 'parcel').astype(int)
c_prob = cross_val_predict(LogisticRegression(max_iter=2000), X, is_parcel,
                           cv=StratifiedKFold(5, shuffle=True, random_state=123),
                           method='predict_proba')[:, 1]
print(f'Predicting COHORT from the features: AUC {roc_auc_score(is_parcel, c_prob):.3f}')
print('(1.0 = the two datasets are perfectly distinguishable, i.e. a strong batch effect;')
print(' 0.5 = indistinguishable, no batch effect.)')

Predicting COHORT from the features: AUC 1.000
(1.0 = the two datasets are perfectly distinguishable, i.e. a strong batch effect;
 0.5 = indistinguishable, no batch effect.)


### Figure: the PCA plot, saved into the repo

Regenerates the cohort scatter from the combined frozen features so the figure lives
with the code that produced it rather than in a stray session. PC 1 and PC 2 are the
first two of the same components the classifier receives - the printed explained-variance
share says how much of the total spread each one accounts for.

In [ ]:
from sklearn.decomposition import PCA

# Standardise first: PCA follows raw variance, and undivided feature scales would let
# a few high-variance dimensions dictate the axes.
Xs = (X - X.mean(0)) / (X.std(0) + 1e-8)
pca = PCA(n_components=2, random_state=123)
Z = pca.fit_transform(Xs)
ev = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(9, 5.5))
for c, colour, lab in (('osf', '#4E79A7', f'OSF - sensor level ({(source=="osf").sum()})'),
                       ('parcel', '#F28E2B', f'Parcel - source level ({(source=="parcel").sum()})')):
    m = source == c
    ax.scatter(Z[m, 0], Z[m, 1], s=34, c=colour, label=lab,
               alpha=0.85, linewidths=0.5, edgecolors='white', zorder=3)
ax.set_xlabel(f'PC 1 ({ev[0]*100:.1f}% of variance)')
ax.set_ylabel(f'PC 2 ({ev[1]*100:.1f}% of variance)')
ax.set_title('The two cohorts occupy separate regions of feature space', fontweight='bold')
ax.grid(alpha=0.3); ax.set_axisbelow(True)
ax.legend(fontsize=9, frameon=False)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
plt.tight_layout()
fname = f'combined_pca_cohorts_{MODEL}.png'
plt.savefig(f'{OUT}/{fname}', dpi=150, bbox_inches='tight'); plt.show()
print('saved', fname)
print(f'PC1 {ev[0]*100:.1f}% | PC2 {ev[1]*100:.1f}% | first 2 together {ev.sum()*100:.1f}%')

## Reading the output

- **Cheat ceiling near 0.5** - the cohorts are balanced, and the combined results in
  notebooks 04 and 05 stand as reported. The batch effect is still real but harmless
  for this particular claim.
- **Cheat ceiling well above 0.5** - report per-cohort numbers alongside the pooled
  one, and compare the pooled number against the ceiling rather than against chance.
- **Per-cohort scores holding up** - the result is real, whichever way the ceiling
  came out.
- **Per-cohort scores collapsing toward 0.5 while the pooled number stays high** -
  the pooled result is an artifact and should not go in the paper as-is.

Whatever the outcome, the cohort-prediction AUC in the last cell is worth a line in the
paper: it documents the domain gap between sensor-level and source-level inputs.